<a href="https://colab.research.google.com/github/Yike-Ding/structural_bioinformatics/blob/main/03_molecular_docking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**03 Molecular Docking**

download assigned pdb code from github respository

In [22]:
import os
import urllib.request


github_pdb_code_url = "https://github.com/Yike-Ding/structural_bioinformatics/blob/main/data/3AFA_H3.1pdb_code"
filename = "3AFA_H3.1pdb_code"

if not os.path.exists(filename):
  print(f"Downloading {filename} from GitHub...")
  urllib.request.urlretrieve(github_pdb_code_url, filename)
  print("Download completed!")
else:
  print(f"{filename} already exists.")

3AFA_H3.1pdb_code already exists.


In [23]:
GITHUB_RAW_PDB_URL = "https://raw.githubusercontent.com/Yike-Ding/structural_bioinformatics/main/data/3AFA_H3.1.pdb"
pdb_path = "3AFA_H3.1.pdb"

if not os.path.exists(pdb_path):
  print(f"Downloading {pdb_path} from GitHub...")
  urllib.request.urlretrieve(GITHUB_RAW_PDB_URL, pdb_path)
  print("Download completed!")
else:
  print(f"{pdb_path} already exists.")

3AFA_H3.1.pdb already exists.


convert .pdb file to .pdbqt format for furture docking

In [24]:
%pip install openbabel
%pip install RDKit

from openbabel import pybel

def pdb_to_pdbqt(pdb_path, pdbqt_path, pH=7.4):
    """
    Convert a PDB file to a PDBQT file needed by docking programs of the AutoDock family.

    Parameters
    ----------
    pdb_path: str or pathlib.Path
        Path to input PDB file.
    pdbqt_path: str or pathlib.path
        Path to output PDBQT file.
    pH: float
        Protonation at given pH.
    """
    molecule = list(pybel.readfile("pdb", str(pdb_path)))[0]
    # add hydrogens at given pH
    molecule.OBMol.CorrectForPH(pH)
    molecule.addh()
    # add partial charges to each atom
    for atom in molecule.atoms:
        atom.OBAtom.GetPartialCharge()
    molecule.write("pdbqt", str(pdbqt_path), overwrite=True)
    return


In [25]:
pdbqt_path = "3AFA_H3.1.pdbqt"
pdb_to_pdbqt(pdb_path, pdbqt_path, pH=7.4)

Due to the fact that there has not published histone H3.1 ligands, several common small molecules are used for docking experiment.

In [26]:

smiles_list = {
    "aspirin": "CC(=O)OC1=CC=CC=C1C(=O)O",
    "caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "ibuprofen": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"
}

for name, smi in smiles_list.items():
    mol = pybel.readstring("smi", smi)
    mol.make3D()
    mol.write("pdbqt", f"{name}.pdbqt", overwrite=True)
    print(f"Test small molecule: {name}.pdbqt")

Test small molecule: aspirin.pdbqt
Test small molecule: caffeine.pdbqt
Test small molecule: ibuprofen.pdbqt


Retrieve the three potential pockets are listed from 02 binding site identification.

In [27]:
pockets = {
    "P_1": {"center": [-42.63, -23.87, 52.24], "size": [20.7, 18.5, 21.8]},
    "P_2": {"center": [-38.91, -8.00, 45.59], "size": [20.6, 25.2, 21.3]},
    "P_3": {"center": [-39.67, -10.12, 47.91], "size": [16.4, 21.9, 15.3]},
}

Install AutoDock Vina.

In [28]:
import urllib.request

vina_url = "https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.7/vina_1.2.7_linux_x86_64"

if not os.path.exists("vina"):
    urllib.request.urlretrieve(vina_url, "vina")

os.chmod("vina", 0o755)   # make the downloaded file executable

Dock each ligand into each pocket. The score is the first `REMARK VINA RESULT` line in the output file (more negative = better).

In [29]:
import subprocess

os.makedirs("poses", exist_ok=True)
scores = {}

for ligand in smiles_list:
    for pocket, (center, size) in pockets.items():
        out = f"poses/{ligand}_{pocket}.pdbqt"
        command = (f"./vina --receptor 3AFA_H3.1.pdbqt --ligand {ligand}.pdbqt "
                   f"--center_x {center[0]} --center_y {center[1]} --center_z {center[2]} "
                   f"--size_x {size[0]} --size_y {size[1]} --size_z {size[2]} "
                   f"--exhaustiveness 8 --out {out}")

        result = subprocess.run(command, shell=True, capture_output=True, text=True)
        if result.returncode != 0:      # Vina failed: show why instead of continuing
            raise RuntimeError(result.stderr)

        for line in open(out):
            if line.startswith("REMARK VINA RESULT"):
                scores[(ligand, pocket)] = float(line.split()[3])
                break
        print(ligand, pocket, scores[(ligand, pocket)])

RuntimeError: Command line parse error: the argument ('c') for option '--center_x' is invalid

Correct usage:

Input:
  --receptor arg             rigid part of the receptor (PDBQT)
  --flex arg                 flexible side chains, if any (PDBQT)
  --ligand arg               ligand (PDBQT)
  --batch arg                batch directory or ligands (PDBQT)
  --scoring arg (=vina)      scoring function (ad4, vina or vinardo)

Search space (required):
  --maps arg                 affinity maps for the autodock4.2 (ad4) or vina 
                             scoring function
  --center_x arg             X coordinate of the center (Angstrom)
  --center_y arg             Y coordinate of the center (Angstrom)
  --center_z arg             Z coordinate of the center (Angstrom)
  --size_x arg               size in the X dimension (Angstrom)
  --size_y arg               size in the Y dimension (Angstrom)
  --size_z arg               size in the Z dimension (Angstrom)
  --autobox                  set

Visualise the best pose in pocket.

In [ ]:
%pip install py3Dmol

import py3Dmol

best = min(scores, key=scores.get)
print(best, scores[best], "kcal/mol")

# keep only the first (best) pose in the output file
pose = ""
for line in open(f"poses/{best[0]}_{best[1]}.pdbqt"):
    pose += line
    if line.startswith("ENDMDL"):
        break

view = py3Dmol.view(width=500, height=400)
view.addModel(open("3AFA_H3.1.pdb").read(), "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "lightgrey"}})
view.addModel(pose, "pdbqt")
view.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon"}})
view.zoomTo({"model": 1})
view.show()